In [1]:
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderUnavailable
import overpy
import overpass
import osm2geojson

import time

import pyperclip
import geojson
import shapely.geometry as geometry
from shapely.ops import linemerge, unary_union, polygonize

import matplotlib.pyplot as plt
import matplotlib

import networkx as nx
import pandas as pd
import numpy as np
import pickle

import matplotlib.pyplot as plt
from sklearn.cluster import SpectralClustering
from sklearn.cluster import KMeans
import node2vec

import folium
import json
from shapely.geometry import mapping

from haversine import haversine

from convenient_pickle import *

from random import shuffle

from area import area

import os
import time

C:\Users\samue\anaconda3\envs\cs7280_env_metaldata\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\samue\anaconda3\envs\cs7280_env_metaldata\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
origin_directory = os.getcwd()

In [3]:
def load_pickles(prefix):
    G = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_graph.pkl')
    total_result_nodes = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_total_result_nodes.pkl')
    total_result_ways = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_total_result_ways.pkl')
    used_bboxes = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_used_bboxes.pkl')
    return G, total_result_nodes, total_result_ways, used_bboxes

In [4]:
G, total_result_nodes, total_result_ways, used_bboxes = load_pickles('06_26_2026')

In [5]:
def convert_to_borders(ways):
    lss = [] 
    
    for ii_w,way in enumerate(ways):
        ls_coords = []
    
        for node in way.nodes:
            ls_coords.append((node.lon,node.lat)) 
    
        lss.append(geometry.LineString(ls_coords))
    
    
    merged = linemerge([*lss]) 
    borders = unary_union(merged) # linestrings to a MultiLineString
    polygons = list(polygonize(borders))
    return merged, borders, polygons

def convert_to_dispersed_borders(ways):
    lss = [] 
    
    for ii_w,way in enumerate(ways):
        ls_coords = []
    
        for node in way.nodes:
            ls_coords.append((node.lon,node.lat)) 
    
        lss.append(geometry.LineString(ls_coords))
    
    
    merged = linemerge([*lss]) 
    return merged
    
#See how many ways each node belongs to
def collect_overlap(ways):
    outdict = dict()
    for way in ways: 
        way_nodes = way.nodes
        for node in way_nodes: 
            if node.id not in outdict.keys(): 
                outdict[node.id] = 1
            else: 
                outdict[node.id] += 1
    outdict = [(i, outdict[i]) for i in outdict.keys()]
    outdict = sorted(outdict, key = lambda x: -x[1])
    return outdict

def make_graph(ways):
    G = nx.Graph()
    
    for way in ways: 
        for node_dex in range(len(way.nodes)): 
            node = way.nodes[node_dex]
            node_id = node.id
            if G.has_node(node_id) == False:
                G.add_node(node_id)
            if node_dex > 0: 
                source = {'lat':node.lat, 'lon':node.lon}
                target = {'lat':way.nodes[node_dex-1].lat, 'lon':way.nodes[node_dex-1].lon}
                edge_distance = get_edge_distance(source, target)
                G.add_edge(node_id,way.nodes[node_dex-1].id,weight=edge_distance)
                
    return G

def make_new_lcc(G, ways):
    cclist = sorted([i for i in nx.connected_components(G)], key = lambda x: -len(x))
    lcc = cclist[0]
    lcc_ways = []
    lcc_nodes = []
    for way in ways: 
        for node in way.nodes: 
            if node.id in lcc: 
                lcc_ways.append(way)
                break
    lcc_node_set = set()
    for way in lcc_ways:
        for node in way.nodes:
            lcc_node_set.add(node.id)
    return pd.Series(lcc_ways), lcc_node_set

# Serialize your merged network to GeoJSON
def quick_map(lcc_merged):
    bike_geojson = mapping(lcc_merged)
    
    # Illinois center
    m = folium.Map(location=[40.0, -89.2], zoom_start=6, tiles='CartoDB positron')
    
    # Illinois outline
    folium.GeoJson(
        "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json",
        name="Illinois",
        style_function=lambda f: {
            'fillColor': '#E6F1FB',
            'color': '#185FA5',
            'weight': 1.5,
            'fillOpacity': 0.25
        } if f['properties']['name'] == 'Illinois' else {
            'fillOpacity': 0,
            'color': 'none',
            'weight': 0
        }
    ).add_to(m)
    
    # Bike/pedestrian network
    folium.GeoJson(
        bike_geojson,
        name="Bike network",
        style_function=lambda f: {
            'color': '#1D9E75',
            'weight': 2,
            'opacity': 0.85
        }
    ).add_to(m)
    
    folium.Rectangle(
        bounds=[[bbox[0], bbox[1]], [bbox[2], bbox[3]]],
        color='#E24B4A',
        weight=2,
        fill=False
    ).add_to(m)
    
    # new_bbox = get_new_bbox(bbox, interval, 'northeast')
    
    # folium.Rectangle(
    #     bounds=[[new_bbox[0], new_bbox[1]], [new_bbox[2], new_bbox[3]]],
    #     color='#E24B4A',
    #     weight=2,
    #     fill=False
    # ).add_to(m)
    
    
    # Zoom to the network
    bounds = lcc_merged.bounds  # (minx, miny, maxx, maxy)
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    
    folium.LayerControl().add_to(m)
    return m

def break_down_into_nodes(ways): 
    node_list = []
    for way in ways: 
        node_list += way.nodes
    return node_list

def get_node_ids(nodelist):
    id_list = []
    for node in nodelist: 
        id_list.append(node.id)
    return id_list

def get_way_node_ids(ways): 
    node_list = break_down_into_nodes(ways)
    id_list = get_node_ids(node_list)
    return id_list

def average_degree(subgraph): 
    average = 0
    for node in subgraph.nodes: 
        average += subgraph.degree(node)
    return average/len(subgraph.nodes)

def get_node_latlon_dict(ways): 
    outdict = dict()
    nodelist = break_down_into_nodes(ways)
    for node in nodelist: 
        outdict[node.id] = {'lat': node.lat, 'lon': node.lon}
    return outdict

def get_edge_distance(node1, node2): 
    lat1 = float(node1['lat'])
    lon1 = float(node1['lon'])
    lat2 = float(node2['lat'])
    lon2 = float(node2['lon'])

    return haversine((lat1,lon1), (lat2, lon2))

def get_graph_distances(use_graph, ways):
    node_latlon_dict = get_node_latlon_dict(ways)
    edges = use_graph.edges
    out_dist = 0
    for edge in edges: 
        node1 = node_latlon_dict[edge[0]]
        node2 = node_latlon_dict[edge[1]]
        out_dist += get_edge_distance(node1, node2)
    return out_dist

def get_degree_distribution(use_graph): 
    outdict = dict()
    for node in use_graph.nodes: 
        degree = use_graph.degree(node)
        if degree not in outdict.keys(): 
            outdict[degree] = 1
        else: 
            outdict[degree] += 1
    return outdict

def find_intersections(outdict): 
    intersections = 0
    for key in outdict.keys(): 
        if key >= 3: 
            intersections += outdict[key]
    return intersections

# Removes all nodes that have a degree of 2, connects intersections/dead ends directly to each other
def strip_the_paths(use_ingraph):
    ingraph = use_ingraph.copy()
    nodes = list(ingraph.nodes)
    for node in nodes: 
        degree = ingraph.degree(node)
        if degree == 2: 
            neighbors = [i for i in ingraph.neighbors(node)]
            ingraph.remove_node(node)
            ingraph.add_edge(neighbors[0], neighbors[1])
    return ingraph

def get_community_box_area(inshape): 
    box = inshape.bounds
    topleft = (box[0],box[1])
    topright = (box[0],box[3])
    bottomleft = (box[2],box[1])
    bottomright = (box[2],box[3])
    polygon = geometry.box(box[0], box[1], box[2], box[3])
    return area(mapping(polygon))/1000000

def get_community_hull_area(inshape): 
    return area(mapping(inshape.convex_hull))/1000000

def get_edge_distance(node1, node2): 
    lat1 = float(node1['lat'])
    lon1 = float(node1['lon'])
    lat2 = float(node2['lat'])
    lon2 = float(node2['lon'])

    return haversine((lat1,lon1), (lat2, lon2))

In [6]:
G = make_graph(total_result_ways)

lcc_ways, lcc_nodes = make_new_lcc(G, total_result_ways)

In [7]:
# After the loop, rebuild:
lcc_merged = convert_to_borders(lcc_ways)[0]
bike_geojson = mapping(lcc_merged)

# Greedy modularity maximization

In [8]:
# Go through each way
# check each node: what community does it belong to?
# If 2 or more nodes belong to the same community, add that way to the output community. 

def run_greedy_modularity(G, lcc_nodes, lcc_ways,resolution = 1,weighted=True,use_n = 5):
    lcc_subgraph = G.subgraph(lcc_nodes)
    if weighted: 
        communities = nx.community.greedy_modularity_communities(lcc_subgraph, resolution=resolution, weight='weight', best_n = n, cutoff=n)
    else: 
        communities = nx.community.greedy_modularity_communities(lcc_subgraph, resolution=resolution, best_n=n, cutoff=n)
    communities = sorted(communities, key = lambda x: -len(x))
    
    community_geojson_dict = dict()
    
    for lcc_way_dex in range(len(lcc_ways)):
    #for lcc_way_dex in range(10):
        #print(f"I am on {lcc_way_dex}/{len(lcc_ways)}")
        lcc_way = lcc_ways[lcc_way_dex]
        count_community_dict = dict() 
        for node in lcc_way.nodes: 
            for community_dex in range(len(communities)):
                use_community = communities[community_dex]
                if node.id in use_community: 
                    if community_dex not in count_community_dict.keys(): 
                        count_community_dict[community_dex] = 1
                    else: 
                        count_community_dict[community_dex] += 1
                    break
            # if count_community_dict[community_dex] >= 2: 
            #     print(count_community_dict)
            #     if community_dex not in community_geojson_dict.keys(): 
            #         community_geojson_dict[community_dex] = [lcc_way]
            #     else: 
            #         community_geojson_dict[community_dex].append(lcc_way)
            #     break
        highest_list = sorted([(i, count_community_dict[i]) for i in count_community_dict.keys()], key = lambda x: -x[1])
        highest = highest_list[0][0]
        if highest not in community_geojson_dict.keys(): 
            community_geojson_dict[highest] = [lcc_way]
        else: 
            community_geojson_dict[highest].append(lcc_way)

    use_map_community_dict = dict()
    for map_community_dex in range(len(list(community_geojson_dict.keys()))):
        use_map_community = community_geojson_dict[map_community_dex] #get the geojson dict from the map_community_dex
        use_map_community_merged = convert_to_borders(use_map_community)[0] #convert that to borders directly
        use_map_community_dict[map_community_dex] = {'shape': use_map_community_merged, 'community': use_map_community}
    return use_map_community_dict, community_geojson_dict


# Analysis of greedy-modularity-maximization communities

In [9]:
def break_down_into_nodes(ways): 
    node_list = []
    for way in ways: 
        node_list += way.nodes
    return node_list

def get_node_ids(nodelist):
    id_list = []
    for node in nodelist: 
        id_list.append(node.id)
    return id_list

def get_way_node_ids(ways): 
    node_list = break_down_into_nodes(ways)
    id_list = get_node_ids(node_list)
    return id_list

def average_degree(subgraph): 
    average = 0
    for node in subgraph.nodes: 
        average += subgraph.degree(node)
    return average/len(subgraph.nodes)

def get_node_latlon_dict(ways): 
    outdict = dict()
    nodelist = break_down_into_nodes(ways)
    for node in nodelist: 
        outdict[node.id] = {'lat': node.lat, 'lon': node.lon}
    return outdict

def get_edge_distance(node1, node2): 
    lat1 = float(node1['lat'])
    lon1 = float(node1['lon'])
    lat2 = float(node2['lat'])
    lon2 = float(node2['lon'])

    return haversine((lat1,lon1), (lat2, lon2))

def get_graph_distances(use_graph, ways):
    node_latlon_dict = get_node_latlon_dict(ways)
    edges = use_graph.edges
    out_dist = 0
    for edge in edges: 
        node1 = node_latlon_dict[edge[0]]
        node2 = node_latlon_dict[edge[1]]
        out_dist += get_edge_distance(node1, node2)
    return out_dist

def get_degree_distribution(use_graph): 
    outdict = dict()
    for node in use_graph.nodes: 
        degree = use_graph.degree(node)
        if degree not in outdict.keys(): 
            outdict[degree] = 1
        else: 
            outdict[degree] += 1
    return outdict

def find_intersections(outdict): 
    intersections = 0
    for key in outdict.keys(): 
        if key >= 3: 
            intersections += outdict[key]
    return intersections

# Removes all nodes that have a degree of 2, connects intersections/dead ends directly to each other
def strip_the_paths(use_ingraph):
    ingraph = use_ingraph.copy()
    nodes = list(ingraph.nodes)
    for node in nodes: 
        degree = ingraph.degree(node)
        if degree == 2: 
            neighbors = [i for i in ingraph.neighbors(node)]
            ingraph.remove_node(node)
            ingraph.add_edge(neighbors[0], neighbors[1])
    return ingraph

def get_community_box_area(inshape): 
    box = inshape.bounds
    topleft = (box[0],box[1])
    topright = (box[0],box[3])
    bottomleft = (box[2],box[1])
    bottomright = (box[2],box[3])
    polygon = geometry.box(box[0], box[1], box[2], box[3])
    return area(mapping(polygon))/1000000

def get_community_hull_area(inshape): 
    return area(mapping(inshape.convex_hull))/1000000

In [10]:
def run_analysis(use_map_community_dict,G): 
    av_degree_dict = dict()
    av_neighbor_degree_dict = dict()
    deg_assort_coeff_dict = dict()
    edge_dict = dict()
    bridge_dict = dict()
    bridge_outta_edges_dict = dict()
    total_distance_dict = dict()
    num_intersections_dict = dict()
    box_area_dict = dict()
    convex_area_dict = dict()
    deadend_dict = dict()
    actual_bridges_dict = dict()
    clustering_dict = dict()
    
    
    
    lcc_subgraph = G.subgraph(lcc_nodes)
    for key in use_map_community_dict.keys():
        current_umc = use_map_community_dict[key]['community']
        umc_subgraph = lcc_subgraph.subgraph(get_way_node_ids(current_umc))
        av_degree_dict[key] = average_degree(umc_subgraph)
        #av_neighbor_degree_dict[key] = nx.average_neighbor_degree(umc_subgraph)
        deg_assort_coeff_dict[key] = nx.assortativity.degree_assortativity_coefficient(umc_subgraph)
        edge_dict[key] = len(umc_subgraph.edges)
        total_distance_dict[key] = get_graph_distances(umc_subgraph, current_umc)
        num_intersections_dict[key] = find_intersections(get_degree_distribution(umc_subgraph))
        box_area_dict[key] = get_community_box_area(use_map_community_dict[key]['shape'])
        convex_area_dict[key] = get_community_hull_area(use_map_community_dict[key]['shape'])
        clustering_dict[key] = nx.average_clustering(umc_subgraph)
    
        only_intersections_and_ends = strip_the_paths(umc_subgraph) 
        bridge_dict[key] = len([i for i in nx.bridges(only_intersections_and_ends)])
        deadend_dict[key] = len([i for i in only_intersections_and_ends.nodes if only_intersections_and_ends.degree(i) == 1])
        actual_bridges_dict[key] = bridge_dict[key] #- deadend_dict[key]
    
        print(f"Done with {key}")
    
    intersections_per_distance = {key: num_intersections_dict[key]/total_distance_dict[key] for key in total_distance_dict.keys()}
    bridges_per_distance = {key: bridge_dict[key]/total_distance_dict[key] for key in total_distance_dict.keys()}
    miles_per_box_area = {key: total_distance_dict[key]/box_area_dict[key] for key in box_area_dict.keys()}
    miles_per_convex_hull_area = {key: total_distance_dict[key]/convex_area_dict[key] for key in convex_area_dict.keys()}
    actual_bridges_per_convex_hull_area = {key: actual_bridges_dict[key]/convex_area_dict[key] for key in convex_area_dict.keys()}
    actual_bridges_per_distance = {key: actual_bridges_dict[key]/total_distance_dict[key] for key in total_distance_dict.keys()}

    out_dict = {'av_degree_dict': av_degree_dict, 'av_neighbor_degree_dict': av_neighbor_degree_dict,
               'deg_assort_coeff_dict':deg_assort_coeff_dict,'edge_dict':edge_dict,
               'bridge_dict':bridge_dict,'bridge_outta_edges_dict':bridge_outta_edges_dict,
               'total_distance_dict':total_distance_dict,'num_intersections_dict':num_intersections_dict,
               'box_area_dict':box_area_dict,'convex_area_dict':convex_area_dict,
                'deadend_dict':deadend_dict,'actual_bridges_dict':actual_bridges_dict,
                'clustering_dict':clustering_dict,'intersections_per_distance':intersections_per_distance,
                'bridges_per_distance':bridges_per_distance,'miles_per_box_area':miles_per_box_area,
                'actual_bridges_per_convex_hull_area':actual_bridges_per_convex_hull_area,
                'actual_bridges_per_distance':actual_bridges_per_distance,
                'miles_per_convex_hull_area':miles_per_convex_hull_area
               }
    return out_dict

def map_those_layers(color_palette, community_geojson_dict):
    # Illinois center
    m = folium.Map(location=[40.0, -89.2], zoom_start=6, tiles='CartoDB positron')
    
    hulls = []
    
    # Illinois outline
    folium.GeoJson(
        "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json",
        name="Illinois",
        style_function=lambda f: {
            'fillColor': '#E6F1FB',
            'color': '#185FA5',
            'weight': 1.5,
            'fillOpacity': 0.25
        } if f['properties']['name'] == 'Illinois' else {
            'fillOpacity': 0,
            'color': 'none',
            'weight': 0
        }
    ).add_to(m)
    
    # Bike/pedestrian network
    
    
    hull_and_community_dict = dict()
    community_shape_dict = dict()
    
    for map_community_double_dex in range(len(list(community_geojson_dict.keys()))):
        hull_and_community = dict()
        map_community_dex=map_community_double_dex
        use_map_community = community_geojson_dict[map_community_dex]
        
        use_map_community_merged = convert_to_dispersed_borders(use_map_community)
    
    
        community_shape_dict[map_community_double_dex] = use_map_community_merged

        um_bike_geojson = mapping(use_map_community_merged)
        test_color_palette = color_palette[map_community_double_dex % len(color_palette)]
        print(test_color_palette)
        folium.GeoJson(
            um_bike_geojson,
            name="Bike network",
            style_function=lambda f, color=test_color_palette: {
                'color': color,
                'weight': 2,
                'opacity': 1
            },
            popup=folium.Popup(f"Community #{map_community_double_dex}")
        ).add_to(m)
    
    
    # Zoom to the network
    try:
        bounds = lcc_merged.bounds  # (minx, miny, maxx, maxy)
        m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    except: 
        pass
        #print("No bounds found")
    
    folium.LayerControl().add_to(m)
    return m
    

def map_those_layers_with_polygons(color_palette, community_geojson_dict):
    # Illinois center
    m = folium.Map(location=[40.0, -89.2], zoom_start=6, tiles='CartoDB positron')
    
    hulls = []
    
    # Illinois outline
    folium.GeoJson(
        "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json",
        name="Illinois",
        style_function=lambda f: {
            'fillColor': '#E6F1FB',
            'color': '#185FA5',
            'weight': 1.5,
            'fillOpacity': 0.25
        } if f['properties']['name'] == 'Illinois' else {
            'fillOpacity': 0,
            'color': 'none',
            'weight': 0
        }
    ).add_to(m)
    
    # Bike/pedestrian network
    
    
    hull_and_community_dict = dict()
    community_shape_dict = dict()
    
    for map_community_double_dex in range(len(list(community_geojson_dict.keys()))):
        hull_and_community = dict()
        map_community_dex=map_community_double_dex
        use_map_community = community_geojson_dict[map_community_dex]
        
        use_map_community_merged = convert_to_dispersed_borders(use_map_community)
    
    
        community_shape_dict[map_community_double_dex] = use_map_community_merged

        um_bike_geojson = mapping(use_map_community_merged)
        test_color_palette = color_palette[map_community_double_dex % len(color_palette)]
        print(test_color_palette)
        folium.GeoJson(
            um_bike_geojson,
            name="Bike network",
            style_function=lambda f, color=test_color_palette: {
                'color': color,
                'weight': 2,
                'opacity': 1
            },
            popup=folium.Popup(f"Community #{map_community_double_dex}")
        ).add_to(m)
    
    
    # Zoom to the network
    try:
        bounds = lcc_merged.bounds  # (minx, miny, maxx, maxy)
        m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    except: 
        pass
        #print("No bounds found")
    
    folium.LayerControl().add_to(m)
    return m

# Output measures

In [11]:
safe_cwd = os.getcwd()

In [19]:
list(range(6,22,2))

[6, 8, 10, 12, 14, 16, 18, 20]

In [23]:
make_new = True
save_new =True
sweep = True
file_name = '7_communities'
os.chdir(safe_cwd)
cwd = os.getcwd()
resolution = 1
n=7
n_list = list(range(6,22,2))





if make_new:
    if sweep == False:
        use_map_community_dict, community_geojson_dict = run_greedy_modularity(G, lcc_nodes, lcc_ways,resolution=1,weighted=False,use_n=n)
        if save_new: 
            dump_pickle(os.getcwd(), f'/community_info/{file_name}',[use_map_community_dict, community_geojson_dict])
            os.chdir(cwd)
    else: 
        for n in n_list: 
            print(f"Starting on {n} communities")
            current_time = time.time()
            use_map_community_dict, community_geojson_dict = run_greedy_modularity(G, lcc_nodes, lcc_ways,resolution=1,weighted=False,use_n=n)
            print(f"Completed greedy modularity. Time took: {time.time() - current_time}.")
            if save_new: 
                use_file_name = str(n) + '_communities'
                dump_pickle(os.getcwd(), f'/community_info/{use_file_name}',[use_map_community_dict, community_geojson_dict])
                os.chdir(cwd)
else: 
    [use_map_community_dict, community_geojson_dict] = load_pickle(f'community_info/{file_name}')
    os.chdir(cwd)

Starting on 6 communities
Completed greedy modularity. Time took: 871.3408849239349.
Starting on 8 communities
Completed greedy modularity. Time took: 944.2534384727478.
Starting on 10 communities
Completed greedy modularity. Time took: 953.1544637680054.
Starting on 12 communities
Completed greedy modularity. Time took: 976.3906185626984.
Starting on 14 communities
Completed greedy modularity. Time took: 850.0948941707611.
Starting on 16 communities
Completed greedy modularity. Time took: 968.1067459583282.
Starting on 18 communities
Completed greedy modularity. Time took: 742.9013638496399.
Starting on 20 communities
Completed greedy modularity. Time took: 914.4834959506989.


In [24]:
color_palette = ["#000000","#228b22","#00008b","#b03060","#ff0000","#ff8800","#de0087","#33cc33","#33ccaa","#ff00ff","#6495ed"]
#m = map_those_layers(color_palette, community_geojson_dict)
#m

In [25]:
os.getcwd()

'C:\\Users\\samue\\Documents\\trail_project_2026\\modular_megatrail'

In [26]:
try: 
    n_list = n_list
except:  
    n_list = n_list = list(range(6,22,2))


try: 
    sweep=sweep
except: 
    sweep=True


if sweep == False: 
    out_dict = run_analysis(use_map_community_dict,G)
else: 
    for n in n_list: 
        cwd = os.getcwd()
        start = time.time()
        use_file_name = str(n) + '_communities'
        [use_map_community_dict, community_geojson_dict] = load_pickle(f'community_info/{use_file_name}')
        out_dict = run_analysis(use_map_community_dict,G)
        print(f"This bit took {time.time() - start} seconds")
        next_time = time.time()
        csv_dict = make_csv_dict(out_dict)
        print(f"This bit took {time.time() - next_time} seconds")
        final_time = time.time()
        save_files(csv_dict, community_geojson_dict, use_file_name)
        print(f"This took {time.time() - final_time} seconds")
        print(f"So this one file took {time.time() - start} seconds")
        print('-----------')
        os.chdir(cwd)
        

Done with 0
Done with 1
Done with 2
Done with 3
Done with 4
Done with 5
This bit took 654.1080341339111 seconds
This bit took 0.002504110336303711 seconds
This took 96.00223112106323 seconds
So this one file took 750.1256084442139 seconds
-----------
Done with 0
Done with 1
Done with 2
Done with 3
Done with 4
Done with 5
Done with 6
Done with 7
This bit took 878.0110535621643 seconds
This bit took 0.0016117095947265625 seconds
This took 110.62753653526306 seconds
So this one file took 988.6508810520172 seconds
-----------
Done with 0
Done with 1
Done with 2
Done with 3
Done with 4
Done with 5
Done with 6
Done with 7
Done with 8
Done with 9
This bit took 1245.2080204486847 seconds
This bit took 0.0022878646850585938 seconds
This took 699.7498505115509 seconds
So this one file took 1944.985978603363 seconds
-----------
Done with 0
Done with 1
Done with 2
Done with 3
Done with 4
Done with 5
Done with 6
Done with 7
Done with 8
Done with 9
Done with 10
Done with 11
This bit took 931.6024391

In [14]:
def make_csv_dict(out_dict):
    csv_dict = {}
    keys = list(out_dict['total_distance_dict'].keys())

    metrics = [
        ('total_distance','total_distance_dict'),
        ('convex_area','convex_area_dict'),
        ('intersections_per_distance','intersections_per_distance'),
        ('miles_per_convex_hull_area','miles_per_convex_hull_area'),
        ('actual_bridges_per_convex_hull_area','actual_bridges_per_convex_hull_area'),
        ('actual_bridges_per_distance','actual_bridges_per_distance'), 
        ('bridges', 'bridge_dict'),
        ('deadends', 'deadend_dict')
    ]

    for prefix, out_key in metrics:
        for k, v in zip(keys, out_dict[out_key].values()):
            csv_dict[f"{prefix}_{k}"] = v

    return csv_dict

#csv_dict = make_csv_dict(out_dict)

    # out_dict = {'av_degree_dict': av_degree_dict, 'av_neighbor_degree_dict': av_neighbor_degree_dict,
    #            'deg_assort_coeff_dict':deg_assort_coeff_dict,'edge_dict':edge_dict,
    #            'bridge_dict':bridge_dict,'bridge_outta_edges_dict':bridge_outta_edges_dict,
    #            'total_distance_dict':total_distance_dict,'num_intersections_dict':num_intersections_dict,
    #            'box_area_dict':box_area_dict,'convex_area_dict':convex_area_dict,
    #             'deadend_dict':deadend_dict,'actual_bridges_dict':actual_bridges_dict,
    #             'clustering_dict':clustering_dict,'intersections_per_distance':intersections_per_distance,
    #             'bridges_per_distance':bridges_per_distance,'miles_per_box_area':miles_per_box_area,
    #             'actual_bridges_per_convex_hull_area':actual_bridges_per_convex_hull_area,
    #             'actual_bridges_per_distance':actual_bridges_per_distance,
    #             'miles_per_convex_hull_area':miles_per_convex_hull_area
    #            }

In [15]:
def save_files(csv_dict, community_geojson_dict, prefix): 
    cwd = os.getcwd()
    os.chdir('csvs')
    if prefix not in os.listdir(): 
        os.mkdir(prefix)
    os.chdir(prefix)
    pd.DataFrame.from_dict(csv_dict, orient="index").T.to_csv(f"{prefix}_map_info_data.csv")

    os.chdir(cwd)
    os.chdir('geojsons')
    if prefix not in os.listdir(): 
        os.mkdir(prefix) 
    os.chdir(prefix)
    for map_community_double_dex in range(len(list(community_geojson_dict.keys()))):
        map_community_dex=map_community_double_dex
        use_map_community = community_geojson_dict[map_community_dex]
        use_map_community_merged = convert_to_borders(use_map_community)[0]
        um_bike_geojson = mapping(use_map_community_merged)
        with open(f'{prefix}_{map_community_double_dex}.geojson', 'w') as file:
            geojson.dump(um_bike_geojson, file)

In [77]:
save_files(csv_dict, community_geojson_dict, '12_communities')


In [20]:
os.getcwd()

'C:\\Users\\samue\\Documents\\trail_project_2026\\modular_megatrail'

In [61]:
# outx = []
# outy = []

# for way in use_map_community_dict[2]['community']: 
#     for node in way.nodes: 
#         outx.append(node.lat)
#         outy.append(node.lon)

# # for way in use_map_community_dict[1]['community']: 
# #     for node in way.nodes: 
# #         outx.append(node.lat)
# #         outy.append(node.lon)

In [62]:
# from matplotlib.pyplot import scatter

In [66]:
# from matplotlib import pyplot as plt
# #scatter([x for x in outy], [y for y in outx], norm = 'linear')

# fig, ax = plt.subplots(figsize = (9, 6))
# ax.scatter(outy, outx, s=60, alpha=0.7, edgecolors="k")
# plt.axis('equal')

In [65]:
#m